In [ ]:
import numpy as np
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pint import UnitStrippedWarning

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericStocks,
    StocksQuintiles,
    SharesInflowStocks,
    ElectricVehicleBatteries,
    GenericMaterials,
    MaterialIntensities,
    MaterialIntensitiesQuintiles,
    RestOf,
)
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.util import resolve_circular_economy_scenario
from imagematerials.concepts import create_electricity_graph
from imagematerials.electricity.constants import EPG_TECHNOLOGIES_FINAL
from imagematerials.electricity.utils import rebroadcast_type_dims
knowledge_graph_electricity = create_electricity_graph()


warnings.filterwarnings("ignore", category=UnitStrippedWarning)


scenario_list = {
    "SSP2_baseline_base":    ("SSP2_baseline", "base"),
    # "SSP2_baseline_full_ce": ("SSP2_baseline", "resource_efficiency"),
                 }


base_path = Path("..", "data", "raw")
path_output = Path("..", "output")
climate_scenarios = list(scenario_list.keys())

# run model

In [ ]:
# Define the complete timeline, including historic tail
time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

# model_runs = {}
all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {scen_id}")
    print(f"Circular economy scenario: {circular_scen}")

    climate_policy_scenario_dir = base_path / Path("image", climate_scen)
    circular_economy_scenario_dir = resolve_circular_economy_scenario(
        base_path / "circular_economy_scenarios", circular_scen)

    # LOAD SECTOR PREPROCESSING --------------------------------------------------------------------

    # 0. VEHICLES
    vhc_sector = get_preprocessing_data("vehicles",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)
    prep_data_vhc = vhc_sector.prep_data
    vhc_sector = Sector('vehicles', prep_data_vhc)

    # 1. BUILDINGS
    bld_sector = get_preprocessing_data("buildings",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)

    # 2.-6. ELECTRICITY
    elc_sector = get_preprocessing_data("electricity",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                        cache = False,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)
    # includes: elc_gen, elc_grid_lines, elc_grid_add, elc_stor_phs, elc_stor_other

    # 7. REST OF SECTOR
    rest_sector = get_preprocessing_data("rest_of",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                        scenario_name=climate_scen,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)

    # 8. EV BATTERY
    ev_battery_sector = get_preprocessing_data("ev_battery",
                                            base_path,
                                            climate_policy_scenario_dir,
                                            circular_economy_scenario_dir,
                                            resource_efficiency_flags_file=circular_economy_scenario_dir)

    # 9. FOSSIL FUELS
    ff_sector = get_preprocessing_data("fossil_fuels",
                                        base_path,
                                        climate_policy_scenario_dir=climate_policy_scenario_dir,
                                        circular_economy_data_file=circular_economy_scenario_dir,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir) 
    # note: if ff transport should be included add "ff_trp" below + FlagIncludeTransportation in resource_efficiency_flags must be set to "true"

    # 10. HYDROGEN PRODUCTION
    h2_sector = get_preprocessing_data("hydrogen",
                                    base_path,
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dir, cache = False,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir)   

    # RUN MODEL ------------------------------------------------------------------------------------
    factory = ModelFactory(
    [vhc_sector, bld_sector, *elc_sector, rest_sector, ev_battery_sector, *ff_sector, h2_sector], 
    complete_timeline
    ).add(StocksQuintiles, ["buildings"]
    ).add(GenericStocks, ["vehicles",
                          "elc_grid_lines", "elc_grid_add", "elc_stor_phs",
                          "ff_extr","ff_proc","ff_pipe", "ff_str",
                          "hydrogen"]
    ).add(ElectricVehicleBatteries, ["ev_battery"],
          input_sources={
              "stock_by_cohort": "vehicles",
              "inflow": "vehicles",
              "outflow_by_cohort": "vehicles"}
    ).add(SharesInflowStocks, ["elc_stor_other",
                               "elc_gen"]
    ).add(MaterialIntensities, ["elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs", "elc_stor_other",
                                "ff_extr","ff_proc","ff_pipe", "ff_str",
                                "hydrogen"] 
    ).add(GenericMaterials,  "vehicles"
    ).add(MaterialIntensitiesQuintiles, "buildings",
    ).add(RestOf, "rest_of"
    )
    model = factory.finish()

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    # renaming Type coordinates (necessary due to work around within the sub-technology model for electricity generation)
    rebroadcast_type_dims(model.elc_gen, knowledge_graph_electricity, EPG_TECHNOLOGIES_FINAL)

    # SAVE OUTPUT ----------------------------------------------------------------------------------
    # save relevant outputs only to reduce memory usage
    all_output[scen_id] = {
        # MATERIAL inflows
        "inflow_materials": [# 0 - vhc, 1 - ev_battery, 2 - buildings, 3 - elc_gen, 4 - elc_grid_lines, 5 - elc_grid_add, 6 - elc_stor_phs, 7 - elc_stor_other, 8 - rest_of
            model.vehicles["inflow_materials"],
            model.buildings["inflow_materials"],
            model.elc_gen["inflow_materials"], model.elc_grid_lines["inflow_materials"], model.elc_grid_add["inflow_materials"], model.elc_stor_phs["inflow_materials"], model.elc_stor_other["inflow_materials"],
            model.rest_of["inflow_materials"],
            model.ev_battery["inflow_battery_materials"],
            model.fossil_fuels["inflow_materials"],
            model.hydrogen["inflow_materials"],
        ],
        # MATERIAL stocks
        "stock_by_cohort_materials": [
            model.vehicles["stock_by_cohort_materials"],
            model.buildings["stock_by_cohort_materials"],
            model.elc_gen["stock_by_cohort_materials"], model.elc_grid_lines["stock_by_cohort_materials"], model.elc_grid_add["stock_by_cohort_materials"], model.elc_stor_phs["stock_by_cohort_materials"], model.elc_stor_other["stock_by_cohort_materials"],
            model.ev_battery["stock_battery_materials"],
            model.fossil_fuels["stock_by_cohort_materials"],
            model.hydrogen["stock_by_cohort_materials"],
        ], 
        # MATERIAL outflows
        "outflow_by_cohort_materials": [
            model.vehicles["outflow_by_cohort_materials"],
            model.buildings["outflow_by_cohort_materials"],
            model.elc_gen["outflow_by_cohort_materials"], model.elc_grid_lines["outflow_by_cohort_materials"], model.elc_grid_add["outflow_by_cohort_materials"], model.elc_stor_phs["outflow_by_cohort_materials"], model.elc_stor_other["outflow_by_cohort_materials"],
            model.ev_battery["outflow_battery_materials"],
            model.fossil_fuels["outflow_battery_materials"],
            model.hydrogen["outflow_battery_materials"],
        ],
        # PRODUCT inflows
        "inflow": [
            model.vehicles["inflow"],
            model.buildings["inflow"],
            model.elc_gen["inflow"], model.elc_grid_lines["inflow"], model.elc_grid_add["inflow"], model.elc_stor_phs["inflow"], model.elc_stor_other["inflow"],
            model.ev_battery["inflow_battery_kWh"],
            model.fossil_fuels["inflow"],
            model.hydrogen["inflow"],
        ],
        # PRODUCT stocks
        "stocks": [
            model.vehicles["stock_by_cohort"],
            model.buildings["stock_by_cohort"],
            model.elc_gen["stock_by_cohort"], model.elc_grid_lines["stock_by_cohort"], model.elc_grid_add["stock_by_cohort"], model.elc_stor_phs["stock_by_cohort"], model.elc_stor_other["stock_by_cohort"],
            model.ev_battery["stock_battery_kWh"],
            model.fossil_fuels["stock_by_cohort"],
            model.hydrogen["stock_by_cohort"],
        ],
        # PRODUCT outflows
        "outflow": [
            model.vehicles["outflow_by_cohort"],
            model.buildings["outflow_by_cohort"],
            model.elc_gen["outflow_by_cohort"],model.elc_grid_lines["outflow_by_cohort"],model.elc_grid_add["outflow_by_cohort"],model.elc_stor_phs["outflow_by_cohort"],model.elc_stor_other["outflow_by_cohort"],
            model.ev_battery["outflow_battery_kWh"],
            model.fossil_fuels["outflow_by_cohort"],
            model.hydrogen["outflow_by_cohort"],
        ],
    }

    # model_runs[scen_id] = model
    print(f"Finished {scen_id}")

# save/load

In [ ]:
outdir = path_output
# outdir.mkdir(parents=True, exist_ok=True)

pkl_path = outdir / "output_RE_targetyear2030.pkl"
# pkl_path = outdir / "output_elc_vhc_bat_ssp2_CE_slow_narrow_ssp2clpo_08-10-26.pkl"

with open(pkl_path, "wb") as f:
    pickle.dump(all_output, f, protocol=5) #pickle.HIGHEST_PROTOCOL

print("Saved to:", pkl_path)

In [ ]:
pkl1 = Path(path_output, "output_RE_targetyear2025.pkl")
pkl2 = Path(path_output, "output_RE_targetyear2030.pkl")


with open(pkl1, "rb") as f:
    all_output_2025 = pickle.load(f)

with open(pkl2, "rb") as f:
    all_output_2030 = pickle.load(f)

# plots

In [ ]:
var = "inflow_materials"
unit = "kilotonne"

all_inflow_materials_2025 = {}
all_inflow_materials_2030 = {}
# 0 - vhc, 1 - ev_battery, 2 - buildings, 3 - elc_gen, 4 - elc_grid_lines, 5 - elc_grid_add, 6 - elc_stor_phs, 7 - elc_stor_other, 8 - rest_of
sector_list = ["vehicles", "ev_battery", "buildings", "elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs", "elc_stor_other", "rest_of"]
for i in range(9):
    print(i)
    if i==8:
        all_inflow_materials_2025[sector_list[i]] = all_output_2025["SSP2_baseline_full_ce"][var][i].sum(["Region"]).pint.to(unit)
        all_inflow_materials_2030[sector_list[i]] = all_output_2030["SSP2_baseline_full_ce"][var][i].sum(["Region"]).pint.to(unit)
    else:
        all_inflow_materials_2025[sector_list[i]] = all_output_2025["SSP2_baseline_full_ce"][var][i].to_array().sum(["Region", "Type"]).pint.to(unit)
        all_inflow_materials_2030[sector_list[i]] = all_output_2030["SSP2_baseline_full_ce"][var][i].to_array().sum(["Region", "Type"]).pint.to(unit)


In [ ]:


material = "copper"            # change to e.g. "copper"
t_start, t_end = 2015, 2070
ncols = 3                     # number of columns; rows are inferred

sectors = ["vehicles", "buildings", "elc_gen", "elc_grid_lines",
           "elc_grid_add",] #"ev_battery",  "elc_stor_phs", "elc_stor_other", "rest_of"

scenarios = {                 # label: (dict, linestyle)
    "2025": (all_inflow_materials_2025, "-"),
    "2030": (all_inflow_materials_2030, "--"),
}

def get_series(da, material):
    """Select material and time range, sum over extra dims (e.g. BatteryType)."""
    da = da.sel(material=material, time=slice(t_start, t_end))
    extra = [d for d in da.dims if d != "time"]
    if extra:
        da = da.sum(dim=extra)
    vals = da.data
    unit = str(getattr(vals, "units", ""))        # pint unit, if present
    vals = np.asarray(getattr(vals, "magnitude", vals))
    return da["time"].values, vals, unit

n = len(sectors)
ncols = min(ncols, n)
nrows = int(np.ceil(n / ncols))

colors = plt.cm.tab10.colors  # cycles automatically if more than 10 sectors
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.3 * nrows),
                         sharex=True, squeeze=False)
axes = axes.ravel()
fig.suptitle(f"Material inflow: {material}", fontsize=14)

for i, (sector, ax) in enumerate(zip(sectors, axes)):
    unit = ""
    for label, (data, ls) in scenarios.items():
        try:
            time, vals, unit = get_series(data[sector], material)
        except (KeyError, ValueError):
            print(f"Skipping {sector} / {label}: '{material}' not available")
            continue
        ax.plot(time, vals, color=colors[i % len(colors)], linestyle=ls, label=label)

    ax.set_title(sector)
    ax.set_ylabel(unit)
    ax.grid(True, alpha=0.3)
    ax.legend()

# hide unused axes, label x-axis on the lowest visible subplot in each column
for ax in axes[n:]:
    ax.set_visible(False)
for j in range(ncols):
    last = max(k for k in range(n) if k % ncols == j)
    axes[last].xaxis.set_tick_params(labelbottom=True)
    axes[last].set_xlabel("Year")

plt.tight_layout()
plt.show()